# How to Run Experiments with the Experiment Framework

This notebook demonstrates how to use the `Experiment` class to benchmark
shape matching methods on datasets.

We will:
1. Load the FAUST test set using `MeshDataset`
2. Create shape pairs using `PairsDataset`
3. Run an experiment with `FunctionalMapMatcher`
4. Analyze the results

## Setup

In [6]:
from geomfum.dataset.torch import MeshDataset, PairsDataset
from geomfum.experiment import Experiment, ExperimentConfig
from geomfum.matcher import FunctionalMapMatcher, MatcherConfig

## Load the FAUST Test Set

The FAUST dataset contains registered human body meshes. We load the test set
with spectral decomposition pre-computed (needed for functional maps).

In [ ]:
# Path to the FAUST test set
dataset_dir = "../../../datasets/faust/test_set"

# Load meshes with spectral features
# spectral=True computes Laplacian eigenfunctions
# distances=True loads/computes geodesic distance matrices (for evaluation)
mesh_dataset = MeshDataset(
    dataset_dir=dataset_dir,
    spectral=True,
    distances=True,
    correspondences=True,  # FAUST has identity correspondences (same topology)
    k=30,  # Number of eigenfunctions
)

print(f"Loaded {len(mesh_dataset)} meshes")

Loaded 20 meshes


## Create Shape Pairs

We create pairs of shapes to match. For a quick demo, we use random sampling
instead of all possible pairs.

In [3]:
# Create pairs dataset
# pair_mode="all" creates all n*(n-1) pairs
# pair_mode="random" with pairs_ratio samples a subset
pairs_dataset = PairsDataset(
    dataset=mesh_dataset,
    pairs_ratio=1,  # Use 10% of possible pairs for demo
)

print(f"Created {len(pairs_dataset)} pairs")

Created 380 pairs


In [22]:
output = matcher(
    pairs_dataset[0]["source"]["shape"], pairs_dataset[0]["target"]["shape"]
)

In [15]:
print(output.p2p21)

[ 233  348  345 ... 4071 6883 4069]


In [23]:
import polyscope as ps

ps.init()
ps_mesh1 = ps.register_surface_mesh(
    "source_mesh",
    pairs_dataset[0]["source"]["shape"].vertices,
    pairs_dataset[0]["source"]["shape"].faces,
)
ps_mesh2 = ps.register_surface_mesh(
    "target_mesh",
    pairs_dataset[0]["target"]["shape"].vertices,
    pairs_dataset[0]["target"]["shape"].faces,
)
ps_mesh1.add_color_quantity(
    "correspondence",
    pairs_dataset[0]["source"]["shape"].vertices,
)
ps_mesh2.add_color_quantity(
    "correspondence",
    pairs_dataset[0]["source"]["shape"].vertices[output.p2p21],
)
ps.show()

## Configure the Matcher

We use `FunctionalMapMatcher` with default settings, but you can customize
the configuration using `MatcherConfig`.

In [ ]:
# Create matcher with default config
matcher = FunctionalMapMatcher()

# Or customize the config
config = MatcherConfig(
    spectrum_size=30, fmap_size=30, sdp_weight=1.0, lb_weight=1e-2, refiners=[]
)
matcher = FunctionalMapMatcher(config=config)

## Run the Experiment

The `Experiment` class handles:
- Iterating over all pairs
- Computing correspondences with the matcher
- Evaluating metrics (geodesic error, coverage, etc.)
- Aggregating results

In [9]:
# Configure the experiment
config = ExperimentConfig(
    name="FAUST_FunctionalMap",
    bidirectional=False,  # Only compute A->B direction
    progress_bar=True,
)

# Create and run the experiment
experiment = Experiment(
    method=matcher,
    dataset=pairs_dataset,
    config=config,
)

result = experiment.run()

Running FAUST_FunctionalMap:   3%|▎         | 12/380 [01:02<31:59,  5.22s/pair, geo_err=0.0549]


KeyboardInterrupt: 

## Analyze Results

In [ ]:
# View aggregated metrics
print("=" * 50)
print("Aggregated Metrics:")
print("=" * 50)
for metric, value in result.metrics.items():
    if not metric.endswith("_std"):
        std = result.metrics.get(f"{metric}_std", 0)
        print(f"{metric}: {value:.4f} ± {std:.4f}")

In [ ]:
# View per-pair metrics
import numpy as np

geodesic_errors = [
    m["geodesic_error"] for m in result.per_pair_metrics if "geodesic_error" in m
]

print("\nGeodesic Error Statistics:")
print(f"  Min:    {np.min(geodesic_errors):.4f}")
print(f"  Max:    {np.max(geodesic_errors):.4f}")
print(f"  Median: {np.median(geodesic_errors):.4f}")

In [ ]:
# Plot error distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(geodesic_errors, bins=20, edgecolor="black", alpha=0.7)
plt.xlabel("Normalized Geodesic Error")
plt.ylabel("Count")
plt.title("Error Distribution")

plt.subplot(1, 2, 2)
plt.boxplot(geodesic_errors)
plt.ylabel("Normalized Geodesic Error")
plt.title("Error Boxplot")

plt.tight_layout()
plt.show()

## Save Results

You can save the results to a JSON file for later analysis.

In [ ]:
# Save results
# result.save("faust_fmatcher_results.json")

# Load results later
# from geomfum.experiment import ExperimentResult
# loaded_result = ExperimentResult.load("faust_fmatcher_results.json")

## Compare Multiple Methods

You can use `ExperimentSuite` to compare multiple matchers.

In [ ]:
# # Compare different matchers
# methods = {
#     "FunctionalMap": FunctionalMapMatcher(),
#     "Quick": QuickMatcher(),
# }
#
# suite = ExperimentSuite(methods, pairs_dataset)
# all_results = suite.run()
# suite.print_comparison(metrics=["geodesic_error", "coverage"])